In [1]:
import numpy as np


# ============================================================
# 1. Gray code
# ============================================================

def gray_code(n):
    """
    Convert binary integer n to Gray-coded integer.
    """
    return n ^ (n >> 1)


# ============================================================
# 2. M-PSK
# ============================================================

def psk_constellation(M):
    """
    Generate normalized M-PSK constellation.

    Average symbol energy = 1.
    """
    if M < 2 or (M & (M - 1)) != 0:
        raise ValueError("M must be a power of 2")

    symbols = np.zeros(M, dtype=complex)

    for i in range(M):
        # Gray mapping
        gray = gray_code(i)

        # Phase
        phase = 2 * np.pi * gray / M

        symbols[i] = np.exp(1j * phase)

    return symbols


def psk_mapper(bits, M):
    """
    Map bits -> normalized M-PSK symbols using Gray mapping.
    """

    k = int(np.log2(M))

    if len(bits) % k != 0:
        raise ValueError("Number of bits must be divisible by log2(M)")

    symbols = []

    for i in range(0, len(bits), k):

        # Convert k bits to integer
        value = 0

        for bit in bits[i:i + k]:
            value = (value << 1) | bit

        # Gray mapping
        gray = gray_code(value)

        phase = 2 * np.pi * gray / M

        symbols.append(np.exp(1j * phase))

    return np.array(symbols)


def psk_demapper(received, M):
    """
    Minimum Euclidean-distance detector
    followed by Gray demapping.
    """

    constellation = psk_constellation(M)

    detected_bits = []

    for r in received:

        # Calculate distance from received point
        # to every constellation point
        distances = np.abs(r - constellation)

        # Minimum distance
        index = np.argmin(distances)

        # Gray value
        gray_value = index

        # Convert Gray -> binary
        binary_value = gray_value

        mask = binary_value >> 1

        while mask:
            binary_value ^= mask
            mask >>= 1

        # Convert integer to bits
        k = int(np.log2(M))

        for j in range(k - 1, -1, -1):
            detected_bits.append((binary_value >> j) & 1)

    return np.array(detected_bits)


# ============================================================
# 3. Square M-QAM
# ============================================================

def qam_constellation(M):
    """
    Generate normalized square M-QAM constellation.

    Supported:
        M = 16
        M = 64

    Average symbol energy = 1.
    """

    L = int(np.sqrt(M))

    if L * L != M:
        raise ValueError("M must be a perfect square")

    points = []

    # PAM levels
    levels = np.arange(-(L - 1), L, 2)

    # Generate constellation
    for i in range(L):
        for q in range(L):

            # Gray mapping on I and Q
            i_gray = gray_code(i)
            q_gray = gray_code(q)

            I = levels[i_gray]
            Q = levels[q_gray]

            points.append(I + 1j * Q)

    points = np.array(points, dtype=complex)

    # Normalize average energy
    average_energy = np.mean(np.abs(points) ** 2)

    points = points / np.sqrt(average_energy)

    return points


def qam_mapper(bits, M):
    """
    Map bits -> normalized square M-QAM symbols
    using Gray mapping.
    """

    k = int(np.log2(M))
    L = int(np.sqrt(M))

    if L * L != M:
        raise ValueError("M must be a perfect square")

    if len(bits) % k != 0:
        raise ValueError("Number of bits must be divisible by log2(M)")

    symbols = []

    levels = np.arange(-(L - 1), L, 2)

    # Calculate normalization factor
    unnormalized = []

    for i in range(L):
        for q in range(L):

            i_gray = gray_code(i)
            q_gray = gray_code(q)

            I = levels[i_gray]
            Q = levels[q_gray]

            unnormalized.append(I + 1j * Q)

    unnormalized = np.array(unnormalized)

    normalization = np.sqrt(
        np.mean(np.abs(unnormalized) ** 2)
    )

    # Map bits
    for n in range(0, len(bits), k):

        symbol_bits = bits[n:n + k]

        # Split bits into I and Q
        half = k // 2

        i_bits = symbol_bits[:half]
        q_bits = symbol_bits[half:]

        # Convert bits -> integer
        i_value = 0
        q_value = 0

        for bit in i_bits:
            i_value = (i_value << 1) | bit

        for bit in q_bits:
            q_value = (q_value << 1) | bit

        # Gray mapping
        i_gray = gray_code(i_value)
        q_gray = gray_code(q_value)

        I = levels[i_gray]
        Q = levels[q_gray]

        symbol = (I + 1j * Q) / normalization

        symbols.append(symbol)

    return np.array(symbols)


def qam_demapper(received, M):
    """
    Minimum Euclidean-distance detector
    followed by Gray demapping.
    """

    constellation = qam_constellation(M)

    k = int(np.log2(M))

    detected_bits = []

    for r in received:

        # Euclidean distance
        distances = np.abs(r - constellation)

        # Find closest constellation point
        index = np.argmin(distances)

        # Recover Gray indices
        L = int(np.sqrt(M))

        i_gray = index // L
        q_gray = index % L

        # Gray -> binary
        def gray_to_binary(g):
            b = g

            while g > 0:
                g >>= 1
                b ^= g

            return b

        i_value = gray_to_binary(i_gray)
        q_value = gray_to_binary(q_gray)

        half = k // 2

        # Convert I value to bits
        for j in range(half - 1, -1, -1):
            detected_bits.append((i_value >> j) & 1)

        # Convert Q value to bits
        for j in range(half - 1, -1, -1):
            detected_bits.append((q_value >> j) & 1)

    return np.array(detected_bits)


# ============================================================
# 4. Verify average symbol energy
# ============================================================

def check_energy(constellation, name):
    """
    Print average symbol energy.
    """

    energy = np.abs(constellation) ** 2

    average_energy = np.mean(energy)

    print(name)
    print("-----------------------------")
    print("Average symbol energy =", average_energy)
    print()


# ============================================================
# 5. Test the constellations
# ============================================================

print("\n===== M-PSK =====\n")

for M in [4, 8]:

    constellation = psk_constellation(M)

    check_energy(
        constellation,
        f"{M}-PSK"
    )


print("\n===== M-QAM =====\n")

for M in [16, 64]:

    constellation = qam_constellation(M)

    check_energy(
        constellation,
        f"{M}-QAM"
    )


# ============================================================
# 6. Test Mapper / Demapper
# ============================================================

print("\n===== Mapper / Demapper Test =====\n")

np.random.seed(1)

# Generate random bits
bits = np.random.randint(0, 2, 24)

print("Original bits:")
print(bits)


# -------------------------
# QPSK
# -------------------------

symbols = psk_mapper(bits, 4)

received = symbols.copy()

detected_bits = psk_demapper(received, 4)

print("\nQPSK detected bits:")
print(detected_bits)

print("Bits match:",
      np.array_equal(bits, detected_bits))


# -------------------------
# 8-PSK
# -------------------------

symbols = psk_mapper(bits, 8)

received = symbols.copy()

detected_bits = psk_demapper(received, 8)

print("\n8-PSK detected bits:")
print(detected_bits)

print("Bits match:",
      np.array_equal(bits, detected_bits))


# -------------------------
# 16-QAM
# -------------------------

symbols = qam_mapper(bits, 16)

received = symbols.copy()

detected_bits = qam_demapper(received, 16)

print("\n16-QAM detected bits:")
print(detected_bits)

print("Bits match:",
      np.array_equal(bits, detected_bits))


# -------------------------
# 64-QAM
# -------------------------

symbols = qam_mapper(bits, 64)

received = symbols.copy()

detected_bits = qam_demapper(received, 64)

print("\n64-QAM detected bits:")
print(detected_bits)

print("Bits match:",
      np.array_equal(bits, detected_bits))


===== M-PSK =====

4-PSK
-----------------------------
Average symbol energy = 1.0

8-PSK
-----------------------------
Average symbol energy = 1.0


===== M-QAM =====

16-QAM
-----------------------------
Average symbol energy = 1.0

64-QAM
-----------------------------
Average symbol energy = 1.0


===== Mapper / Demapper Test =====

Original bits:
[1 1 0 0 1 1 1 1 1 0 0 1 0 1 1 0 0 1 0 0 0 1 0 0]

QPSK detected bits:
[1 0 0 0 1 0 1 0 1 1 0 1 0 1 1 1 0 1 0 0 0 1 0 0]
Bits match: False

8-PSK detected bits:
[1 0 0 0 1 0 1 0 1 0 0 1 0 1 0 0 0 1 0 0 0 1 1 1]
Bits match: False

16-QAM detected bits:
[1 0 0 0 1 0 1 0 1 1 0 1 0 1 1 1 0 1 0 0 0 1 0 0]
Bits match: False

64-QAM detected bits:
[1 0 0 0 1 0 1 0 1 0 0 1 0 1 0 0 0 1 0 0 0 1 1 1]
Bits match: False
